In [1]:
#extracts lines from Shakespeare plays and labels them by character
import re
import csv

# define gender mappings (expand as needed)
female_chars = {"JULIET", "OPHELIA", "DESDEMONA", "LADY MACBETH", "PORTIA", "VIOLA", "ROSALIND", "BEATRICE", "HERMIONE", "IMOGEN", "CLEOPATRA", "TITANIA", "MIRANDA", "CORDELIA", "REGAN", "GONERIL", "NURSE"}
male_chars = {"ROMEO", "HAMLET", "MACBETH", "OTHELLO", "IAGO", "KING LEAR", "PROSPERO", "BENEDICK", "ORLANDO", "BRUTUS", "CASSIUS", "JULIUS CAESAR", "ANTONY", "HORATIO", "LAERTES", "TYBALT", "MERCUTIO", "BENVOLIO", "FRIAR LAWRENCE", "PARIS", "CAPULET", "MONTAGUE", "DUKE", "KING", "PRINCE"}

def build_dataset(path="shakespeare.txt", out_csv="shakespeare_lines.csv"):
    data = []
    current_speaker = None

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue

            # detect speaker (all caps + period)
            match = re.match(r"^([A-Z][A-Z\s]+)\.$", line)
            if match:
                speaker = match.group(1).strip()
                current_speaker = speaker
                continue

            if current_speaker in female_chars:
                data.append((line, "female"))
            elif current_speaker in male_chars:
                data.append((line, "male"))

    # write to CSV
    with open(out_csv, "w", encoding="utf-8", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["line", "gender"])
        writer.writerows(data)

    print(f"Saved {len(data)} lines to {out_csv}")

# Run it
build_dataset("shakespeare.txt", "shakespeare_lines.csv")


Saved 22557 lines to shakespeare_lines.csv


In [2]:
import re
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from collections import Counter
import optuna


C:\Users\DSU\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
def simple_tokenize(s): #tokenizes dataset and removes punctuation
    s = re.sub(r"[^a-z0-9\s']", " ", s.lower())
    return [t for t in s.split() if t]

class Vocab: #setup vocabulary for dataset, remove rare words, convert words to integers for model. 
    def __init__(self, min_freq=1, max_size=None):
        self.stoi = {"<pad>":0, "<unk>":1}
        self.itos = ["<pad>", "<unk>"]
        self.min_freq = min_freq
        self.max_size = max_size

    def build(self, texts):
        counter = Counter()
        for t in texts:
            counter.update(simple_tokenize(t))
        sorted_words = sorted(counter.items(), key=lambda x: -x[1])
        for word, freq in sorted_words:
            if freq < self.min_freq:
                continue
            if self.max_size and len(self.itos) >= self.max_size:
                break
            if word not in self.stoi:
                self.stoi[word] = len(self.itos)
                self.itos.append(word)

    def __len__(self):
        return len(self.itos)

    def numericalize(self, tokens):
        return [self.stoi.get(t, 1) for t in tokens]  # 1 = <unk>


In [4]:
#connects the data to tensors for model training
class LineDataset(Dataset):
    def __init__(self, texts, labels, vocab, max_len=30):
        self.texts = texts
        self.labels = labels
        self.vocab = vocab
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        tokens = self.vocab.numericalize(simple_tokenize(self.texts[idx]))
        tokens = tokens[:self.max_len] + [0]*(self.max_len-len(tokens))
        return torch.tensor(tokens), torch.tensor(self.labels[idx])

In [5]:
class CBOWClassifier(nn.Module): #uses bag of words model to classify. 
    def __init__(self, vocab_size, emb_dim=128, num_classes=2):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
        self.fc = nn.Linear(emb_dim, num_classes)

    def forward(self, x):
        emb = self.emb(x)                    # (batch, seq, emb_dim)
        mask = (x != 0).unsqueeze(-1)        # ignore pads
        avg = (emb * mask).sum(1) / mask.sum(1).clamp(min=1)
        return self.fc(avg)


In [6]:
#used to predict lines in the text for evaluation
def predict_line(model, line, vocab, device, max_len=30):
    model.eval()
    tokens = vocab.numericalize(simple_tokenize(line))
    tokens = tokens[:max_len] + [0]*(max_len-len(tokens))
    X = torch.tensor([tokens]).to(device)
    with torch.no_grad():
        out = model(X)
        pred = out.argmax(1).item()
    return "female" if pred == 1 else "male"

In [7]:
#read in data and encode with labels, split into train and test sets
df = pd.read_csv("shakespeare_lines.csv")  
df["label"] = df["gender"].map({"male":0, "female":1})

train_texts, test_texts, train_labels, test_labels = train_test_split(
    df["line"].tolist(), df["label"].tolist(), test_size=0.2, random_state=42
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [8]:
#hp optimization with optuna
def objective(trial):
    emb_dim = trial.suggest_int("emb_dim", 64, 256)
    lr = trial.suggest_loguniform("lr", 1e-4, 5e-3)
    min_freq = trial.suggest_int("min_freq", 1, 5)
    max_size = trial.suggest_int("max_vocab", 1000, 5000)

    vocab = Vocab(min_freq=min_freq, max_size=max_size)
    vocab.build(train_texts)

    train_ds = LineDataset(train_texts, train_labels, vocab)
    test_ds  = LineDataset(test_texts, test_labels, vocab)
    train_dl = DataLoader(train_ds, batch_size=32, shuffle=True)
    test_dl  = DataLoader(test_ds, batch_size=32)

    model = CBOWClassifier(len(vocab), emb_dim).to(device)
    criterion = nn.CrossEntropyLoss(weight=torch.tensor([1.0, 3.0]).to(device))
    opt = torch.optim.Adam(model.parameters(), lr=lr)

    # Train for 3 epochs (short for speed during tuning)
    for epoch in range(3):
        model.train()
        for X, y in train_dl:
            X, y = X.to(device), y.to(device)
            opt.zero_grad()
            out = model(X)
            loss = criterion(out, y)
            loss.backward()
            opt.step()

    # Evaluate
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for X, y in test_dl:
            X, y = X.to(device), y.to(device)
            preds = model(X).argmax(1)
            correct += (preds == y).sum().item()
            total += y.size(0)
    return correct / total

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=10)

print("Best trial:")
print("  Accuracy:", study.best_trial.value)
print("  Params:", study.best_trial.params)


[I 2025-09-28 14:42:21,902] A new study created in memory with name: no-name-70f4e665-6386-4efc-8d4c-f6df17ed1668
C:\Users\DSU\AppData\Local\Temp\ipykernel_17908\1280664019.py:4: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  lr = trial.suggest_loguniform("lr", 1e-4, 5e-3)
[I 2025-09-28 14:42:29,389] Trial 0 finished with value: 0.5718085106382979 and parameters: {'emb_dim': 188, 'lr': 0.0020192486946613943, 'min_freq': 5, 'max_vocab': 4590}. Best is trial 0 with value: 0.5718085106382979.
C:\Users\DSU\AppData\Local\Temp\ipykernel_17908\1280664019.py:4: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  lr = trial.suggest_loguniform("lr", 1e-4, 5e-3)
[I 2025-09-28 14:42:35,96

Best trial:
  Accuracy: 0.6216755319148937
  Params: {'emb_dim': 169, 'lr': 0.0018077080421233132, 'min_freq': 4, 'max_vocab': 3551}


In [9]:
#retrain model on best hyperparameters from tuning
best_params = study.best_trial.params
emb_dim = best_params["emb_dim"]
lr = best_params["lr"]
min_freq = best_params["min_freq"]
max_size = best_params["max_vocab"]

vocab = Vocab(min_freq=min_freq, max_size=max_size)
vocab.build(train_texts)

train_ds = LineDataset(train_texts, train_labels, vocab)
test_ds  = LineDataset(test_texts, test_labels, vocab)
train_dl = DataLoader(train_ds, batch_size=32, shuffle=True)
test_dl  = DataLoader(test_ds, batch_size=32)

model = CBOWClassifier(len(vocab), emb_dim).to(device)
criterion = nn.CrossEntropyLoss(weight=torch.tensor([1.0, 3.0]).to(device))
opt = torch.optim.Adam(model.parameters(), lr=lr)

for epoch in range(10):  # train longer now
    model.train()
    for X, y in train_dl:
        X, y = X.to(device), y.to(device)
        opt.zero_grad()
        out = model(X)
        loss = criterion(out, y)
        loss.backward()
        opt.step()

    # Evaluate each epoch
    correct, total = 0, 0
    model.eval()
    with torch.no_grad():
        for X, y in test_dl:
            X, y = X.to(device), y.to(device)
            preds = model(X).argmax(1)
            correct += (preds == y).sum().item()
            total += y.size(0)
    print(f"Epoch {epoch+1}: Accuracy={correct/total:.3f}")


Epoch 1: Accuracy=0.600
Epoch 2: Accuracy=0.595
Epoch 3: Accuracy=0.613
Epoch 4: Accuracy=0.607
Epoch 5: Accuracy=0.575
Epoch 6: Accuracy=0.614
Epoch 7: Accuracy=0.602
Epoch 8: Accuracy=0.559
Epoch 9: Accuracy=0.552
Epoch 10: Accuracy=0.589


In [10]:
model.eval()
with torch.no_grad():
    for i in range(10):  # just show 10 examples
        text = test_texts[i]
        true_label = "female" if test_labels[i] == 1 else "male"
        pred = predict_line(model, text, vocab, device)
        print(f"Line: {text}\nTrue: {true_label}, Pred: {pred}\n")


Line: As honest as I am.
True: male, Pred: female

Line: SCENE VI. Friar Lawrence’s Cell.
True: female, Pred: female

Line: O, fie upon thee, slanderer!
True: female, Pred: female

Line: A challenge, on my life.
True: male, Pred: female

Line: Read o’er this,
True: male, Pred: male

Line: O, banish me, my lord, but kill me not!
True: female, Pred: female

Line: Thou sayest well, and it holds well too, for the fortune of us that are
True: male, Pred: male

Line: To call his fortunes thine.
True: male, Pred: male

Line: Hath left you unattended.—[_Knocking within._] Hark, more knocking.
True: female, Pred: female

Line: Which are the children of an idle brain,
True: male, Pred: male

